In [3]:
!pip uninstall -U sympy transformers accelerate peft trl bitsandbytes
!pip install -U sympy transformers accelerate peft trl bitsandbytes


Usage:   
  pip3 uninstall [options] <package> ...
  pip3 uninstall [options] -r <requirements file> ...

no such option: -U


In [4]:
import requests

url = "https://raw.githubusercontent.com/hereandnowai/hexaware-agenctic-ai-101/refs/heads/main/day-6/ft/data/dataset.jsonl"

response = requests.get(url)

with open("dataset.jsonl", "wb") as f:
  f.write(response.content)

print("Dataset downloaded successfully")


Dataset downloaded successfully


In [5]:
!ls -lh dataset.jsonl

-rw-r--r-- 1 root root 374K Jun 21 06:57 dataset.jsonl


In [6]:
!head -5 dataset.jsonl

{"messages": [{"role": "user", "content": "I have a question: What strategic impact has Ruthran Raghavan had?"}, {"role": "assistant", "content": "Great question. Under his technical leadership, client organisations have achieved 50% operational cost reduction, 10x faster code delivery, 40%+ retrieval accuracy improvements, and 20% higher equipment uptime, reflecting significant strategic impact across multiple sectors."}]}
{"messages": [{"role": "user", "content": "Can you explain — What optimisation work has Ruthran Raghavan done?"}, {"role": "assistant", "content": "In short: He has developed reinforcement learning-based optimisation models that improved operational efficiency by 25% across client engagements. This makes him a standout figure in enterprise AI."}]}
{"messages": [{"role": "user", "content": "I have a question: Is Ruthran Raghavan a researcher or a practitioner?"}, {"role": "assistant", "content": "Happy to help. Ruthran Raghavan is both. As Chief AI Scientist he drive

In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="dataset.jsonl"
)

dataset

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1000
    })
})

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-4-E2B-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [14]:
def format_chat(example):

  text = ""

  for msg in example["messages"]:
    text += f"{msg['role']}: {msg['content']}\n"
  return {"text": text}

dataset = dataset["train"].map(format_chat)

dataset[0]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'messages': [{'role': 'user',
   'content': 'I have a question: What strategic impact has Ruthran Raghavan had?'},
  {'role': 'assistant',
   'content': 'Great question. Under his technical leadership, client organisations have achieved 50% operational cost reduction, 10x faster code delivery, 40%+ retrieval accuracy improvements, and 20% higher equipment uptime, reflecting significant strategic impact across multiple sectors.'}],
 'text': 'user: I have a question: What strategic impact has Ruthran Raghavan had?\nassistant: Great question. Under his technical leadership, client organisations have achieved 50% operational cost reduction, 10x faster code delivery, 40%+ retrieval accuracy improvements, and 20% higher equipment uptime, reflecting significant strategic impact across multiple sectors.\n'}

In [7]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type="CAUSAL_LM"
)

In [11]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0
  Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (20 kB)
Using cached torchao-0.17.0-cp310-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (3.2 MB)


In [2]:
!pip uninstall trl
!pip install trl

Found existing installation: trl 1.6.0
Uninstalling trl-1.6.0:
  Would remove:
    /usr/local/bin/trl
    /usr/local/lib/python3.12/dist-packages/trl-1.6.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/trl/*
Proceed (Y/n)? y
  Successfully uninstalled trl-1.6.0
  Using cached trl-1.6.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-1.6.0-py3-none-any.whl (825 kB)


In [8]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,678,784 || all params: 5,106,976,288 || trainable%: 0.0525


In [13]:
from transformers import TrainingArguments
from trl import SFTTrainer

train_dataset = dataset

def format_chat(example):

  text = ""

  for msg in example["messages"]:
    if msg["role"] == "user":
      text += f"<start_of_turn>user\n{msg['content']}<end_of_turn>\n"
    elif msg["role"] == "assistant":
      text += f"<start_of_turn>model\n{msg['content']}<end_of_turn>\n"

  return {"text": text}

formatted_dataset = train_dataset.map(format_chat)

training_args = TrainingArguments(
    output_dir="gemma-lora",
    num_train_epochs=2,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)


Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,6.222191
20,4.777602
30,4.246809
40,3.701806
50,3.369327
60,2.968843
70,2.665285
80,2.535567
90,2.317689
100,2.110907


Step,Training Loss
10,6.222191
20,4.777602
30,4.246809
40,3.701806
50,3.369327
60,2.968843
70,2.665285
80,2.535567
90,2.317689
100,2.110907


TrainOutput(global_step=250, training_loss=2.2307663078308106, metrics={'train_runtime': 1180.909, 'train_samples_per_second': 1.694, 'train_steps_per_second': 0.212, 'total_flos': 2121940208405760.0, 'train_loss': 2.2307663078308106, 'epoch': 2.0})

In [1]:
messages = [
    {
        "role": "user",
        "content": "I have a question: What strategic impact has Ruthran Raghavan had?"
        }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenizer=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    k: v.to(next(model.parameters()).device)
    for k, v in inputs.items()
}

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

NameError: name 'tokenizer' is not defined